# In-Class Lab 1: Is This Data Good Enough to Build On?
## Talabat — ETA Prediction Data Quality Audit

**Welcome to your first in-class lab!** 🎉

**The scenario:** Talabat's leadership wants to show customers a more accurate estimated delivery time (ETA) at checkout. Before anyone builds a prediction model, they've asked you — the data team — to check whether the order/delivery data can actually be trusted.

**By the end of this notebook, you will be able to:**
- Build a statistical profile of a raw dataset and know what each diagnostic is telling you
- Use systematic rules (not just eyeballing) to flag statistical outliers
- Run and interpret a t-test and a Chi-Square test — and check whether their assumptions actually hold
- Write business-logic checks that catch problems statistics alone would miss
- Judge whether a dataset is actually fit for the business problem it's meant to solve

**Structure:**
| Section | Focus |
|---|---|
| 1 | Statistical Diagnostics |
| 2 | Logical / Business-Rule Checks |
| 3 | Discussion |

**A note on how this notebook works:** most code cells are already written for you. Where you see:
```python
### START CODE HERE ### (~ 1 line of code)
result = None
### END CODE HERE ###
```
that's your part — replace `None` with the actual code. Everything else, just run as-is. Each exercise tells you what output to expect so you can check yourself before moving on.

## Setup — Load the Data

Run the cell below. This part is provided — no changes needed.

In [ ]:
import pandas as pd
import numpy as np
import re
from scipy.stats import chi2_contingency, fisher_exact, shapiro, levene, ttest_ind, mannwhitneyu

# TODO (instructor): replace with the raw GitHub URL once the CSV is uploaded
RAW_URL = "https://raw.githubusercontent.com/<username>/<repo>/main/talabat_eta_orders.csv"

df = pd.read_csv(RAW_URL, parse_dates=["order_time", "promised_delivery_time", "actual_delivery_time"])
df["duration_min"] = (df["actual_delivery_time"] - df["order_time"]).dt.total_seconds() / 60
df.head()

---
# Section 1 — Statistical Diagnostics

This section follows the two diagnostic tables from Lecture 2 directly: first the **basics** (does the data look complete and reasonable at a glance?), then **look deeper** (are there patterns, relationships, or anomalies that only show up once you dig — including two formal statistical tests).

## 1a. Start With the Basics

### Exercise 1 — Missing Values

The first thing to check in any dataset: how much is missing, and where? A column that's 40% missing needs very different treatment than one that's 1% missing.

`df.isna().mean()` gives you the **fraction** of missing values per column. Multiply by 100 (or format as `%`) to read it as a percentage.

In [ ]:
### START CODE HERE ### (~ 1 line of code)
missing_pct = None
### END CODE HERE ###

missing_pct.sort_values(ascending=False)

**Expected output (top 2 rows):**
```
courier_id              0.077
actual_delivery_time    0.054
```
**Question:** every other column shows 0% missing — does that mean they're all trustworthy? _(your answer here)_

### Exercise 2 — Typical Value and Spread

For a numeric column, "typical value" usually means mean or median, and "spread" usually means standard deviation or IQR (Interquartile Range = Q3 − Q1, the middle 50% of the data).

Compute the mean, median, standard deviation, min, and max of `duration_min` in one line using `.agg()`.

In [ ]:
### START CODE HERE ### (~ 1 line of code)
duration_summary = None
### END CODE HERE ###

duration_summary

**Expected output (approximate):**
```
mean      35.2
median    34.6
std       18.1
min        3.0
max      480.0
```
**Question:** the max is 480 minutes (8 hours) and the min is 3 minutes. Do either of those look like real deliveries to you? Hold that thought — Section 1b will let you check systematically.

### Exercise 3 — Quartiles and IQR

Fill in the quartile computation below. `.quantile([0.25, 0.75])` returns Q1 and Q3 as a 2-element Series — unpack it into `q1, q3`.

In [ ]:
### START CODE HERE ### (~ 1 line of code)
q1, q3 = None
### END CODE HERE ###

iqr = q3 - q1
print(f"Q1={q1:.2f}  Q3={q3:.2f}  IQR={iqr:.2f}")

**Expected output:** `Q1=26.51  Q3=42.74  IQR=16.23`

Keep `q1`, `q3`, and `iqr` — you'll reuse them in Exercise 6.

### Exercise 4 — How Are Categories Distributed?

For a categorical column like `zone`, the equivalent of "typical value" is a **frequency table**: how often does each category appear? `value_counts(normalize=True)` gives proportions instead of raw counts.

In [ ]:
### START CODE HERE ### (~ 1 line of code)
zone_freq = None
### END CODE HERE ###

zone_freq.round(3)

**Expected output (top 3):**
```
Nasr City    0.193
Maadi        0.165
Heliopolis   0.161
```
**Question:** scroll to the bottom of your output. Do you see any zone names that look like they should be the same zone, just written differently? Do you see any zone with a suspiciously small share? _(your answer here)_

### Exercise 5 — Cardinality and Duplicates

Two quick checks: **cardinality** (how many distinct values does a column have?) via `.nunique()`, and **duplicate rows** via `.duplicated()`.

In [ ]:
### START CODE HERE ### (~ 2 lines of code)
cardinality = None
duplicate_count = None
### END CODE HERE ###

print(cardinality)
print("\nDuplicate order_id rows:", duplicate_count)

**Expected output:** `restaurant_id: 149`, `courier_id: 200`, `zone: 9`, `Duplicate order_id rows: 0`

**Question:** `zone` shows 9 unique values — but how many *real* zones do you think there actually are, based on what you saw in Exercise 4?

## 1b. Look Deeper

Now that you've done the basics, dig into structure: shape, outliers, relationships, formal statistical tests, and change over time.

### Exercise 6 — Systematic Outlier Detection (IQR Rule)

Eyeballing min/max (like in Exercise 2) doesn't scale, and it's subjective. The **IQR rule** gives a repeatable definition: anything below `Q1 - 1.5×IQR` or above `Q3 + 1.5×IQR` is flagged as a statistical outlier. You already have `q1`, `q3`, and `iqr` from Exercise 3.

In [ ]:
### START CODE HERE ### (~ 2 lines of code)
lower_bound = None
upper_bound = None
### END CODE HERE ###

outliers = df[(df["duration_min"] < lower_bound) | (df["duration_min"] > upper_bound)]
print(f"Bounds: [{lower_bound:.2f}, {upper_bound:.2f}] minutes")
print(f"Rows flagged: {len(outliers)}")
outliers[["order_id", "zone", "distance_km", "duration_min"]]

**Expected output:** bounds `[2.17, 67.07]`, **11 rows flagged**.

### Exercise 7 — Are Variables Related?

`.corr()` gives the Pearson correlation coefficient between two numeric columns, ranging from -1 (perfectly inverse) to +1 (perfectly aligned). We'd expect distance and delivery duration to be positively correlated — farther should generally mean longer.

In [ ]:
### START CODE HERE ### (~ 1 line of code)
correlation = None
### END CODE HERE ###

correlation

**Expected output:** correlation ≈ **0.52**. Positive, but far from 1 — plenty of other factors are clearly affecting duration too. That's expected, not a red flag.

### Exercise 8 — Are Categorical Variables Associated? (Cross-tab + Chi-Square Test)

A cross-tab checks whether two categorical patterns move together. Here: is missing `courier_id` evenly spread across zones, or concentrated in specific ones? Build a boolean column `missing_courier` first, then cross-tab it against `zone` with `normalize="index"` (so each zone's row sums to 1).

In [ ]:
df["missing_courier"] = df["courier_id"].isna()

### START CODE HERE ### (~ 1 line of code)
courier_missing_by_zone = None
### END CODE HERE ###

courier_missing_by_zone.round(3)

**Expected output:** most zones sit around 6–10% missing — except one zone at roughly **35%**.

**Question:** which zone stands out, and connect it back to Exercise 4 — what else did you notice about that zone there? _(your answer here)_

Eyeballing a crosstab is a good start, but it doesn't tell you whether the pattern is *statistically* meaningful or could just be random noise. A **Chi-Square test of independence** answers that formally: it tests whether `zone` and `missing_courier` are truly associated.

In [ ]:
contingency_table = pd.crosstab(df["zone"], df["missing_courier"])

### START CODE HERE ### (~ 1 line of code)
chi2, p_value, dof, expected = None
### END CODE HERE ###

print(f"Chi2 = {chi2:.2f}, p-value = {p_value:.6f}, dof = {dof}")

**Expected output:** `Chi2 = 34.60, p-value = 0.000032, dof = 8` — a very small p-value, suggesting the association is real, not random noise.

**But before trusting that p-value:** the Chi-Square test assumes every *expected* cell count is reasonably large (a common rule of thumb: no more than 20% of cells below 5, and none below 1). Check the expected frequencies table below (already computed above as `expected`) — this is a provided cell, just run it and look closely.

In [ ]:
expected_df = pd.DataFrame(expected, index=contingency_table.index, columns=contingency_table.columns)
expected_df.round(2)

**Question:** which single cell has the smallest expected frequency, and is it below the rule-of-thumb threshold of 5? Does that change how much you'd trust this test's p-value *specifically for that zone*, even if the overall test result is still technically valid? _(your answer here)_

As a robustness check, you can isolate just the zone in question against "everyone else" as a focused 2×2 table and run **Fisher's Exact Test** — a test built specifically for small-sample situations like this one. It's provided in full below.

In [ ]:
df["is_new_zone"] = df["zone"] == "Sheikh Zayed Extension"
focused_table = pd.crosstab(df["is_new_zone"], df["missing_courier"])

odds_ratio, p_fisher = fisher_exact(focused_table)
print(f"Fisher's exact test (Sheikh Zayed Extension vs. everyone else): odds_ratio = {odds_ratio:.2f}, p = {p_fisher:.6f}")

**Expected output:** `p ≈ 0.00008` — confirms the association holds even under a test designed for exactly this small-sample situation.

### Exercise 9 — Do Two Groups Differ? (t-test)

You may have noticed `Sheikh Zayed Extension` deliveries seem to take longer. Is that a real difference, or could it just be chance given how few orders that zone has? A **t-test** answers this formally — comparing the mean `duration_min` of two groups — but only if its assumptions hold.

In [ ]:
new_zone_duration = df[df["zone"] == "Sheikh Zayed Extension"]["duration_min"].dropna()
rest_duration = df[df["zone"] != "Sheikh Zayed Extension"]["duration_min"].dropna()

print(f"New zone: n={len(new_zone_duration)}, mean={new_zone_duration.mean():.2f}, std={new_zone_duration.std():.2f}")
print(f"Rest:     n={len(rest_duration)}, mean={rest_duration.mean():.2f}, std={rest_duration.std():.2f}")

**Expected output:** New zone n=22, mean≈57.66, std≈15.07. Rest n=2840, mean≈35.00, std≈18.01. A ~23-minute gap — worth testing formally.

**Before running the test, check two assumptions:**
1. Each group is roughly normally distributed — use the **Shapiro-Wilk test** (small p-value = *not* normal)
2. The two groups have roughly equal variance — use **Levene's test** (small p-value = *not* equal)

In [ ]:
### START CODE HERE ### (~ 2 lines of code)
normality_check = None
variance_check = None
### END CODE HERE ###

print("Shapiro-Wilk (new zone):", normality_check)
print("Levene's test:", variance_check)

**Expected output:** Shapiro-Wilk gives `p ≈ 0.047` (borderline — right at the edge of "not normal" at the usual 5% cutoff). Levene's gives `p ≈ 0.32` (equal variance looks fine).

**Question:** with the new zone's normality borderline-failing, should you fully trust a standard t-test here? Keep this in mind for the next cell.

Run **Welch's t-test** — a version that doesn't assume equal variance, and is generally the safer default even when Levene's test says variances look equal (pass `equal_var=False`).

In [ ]:
### START CODE HERE ### (~ 1 line of code)
t_stat, p_value = None
### END CODE HERE ###

print(f"Welch's t-test: t = {t_stat:.3f}, p = {p_value:.6f}")

**Expected output:** `t ≈ 7.01, p < 0.000001` — a highly significant difference.

Given the borderline normality result, cross-check with the **Mann-Whitney U test** — a non-parametric alternative that makes no normality assumption at all (this is the "Mann-Whitney" the lecture slide paired with the t-test). It's provided in full.

In [ ]:
u_stat, p_mw = mannwhitneyu(new_zone_duration, rest_duration, alternative="two-sided")
print(f"Mann-Whitney U test: p = {p_mw:.6f}")

**Expected output:** `p < 0.000001` — agrees with the t-test.

**Question:** why is it reassuring that a test making *no* normality assumption reaches the same conclusion as the t-test, given the new zone's borderline Shapiro-Wilk result? _(your answer here)_

### Exercise 10 — Has the Data Changed Over Time?

This one's provided in full — just run it and look closely at the chart.

In [ ]:
df["order_date"] = df["order_time"].dt.date
df.groupby("order_date").size().plot(figsize=(10, 3), title="Orders per day")

**Question:** do you see anything unusual in the timeline — a gap, a spike, a drop? What real-world event might explain it? _(your answer here)_

### Exercise 11 — Unusual Only in Combination

Sometimes no single value looks wrong, but the *combination* does — like the age/income/experience example from the lecture. Here, compute an **implied speed** (distance ÷ time) for every order. A value alone (say, 7 km) or a duration alone (3 min) might look fine individually — but together they can imply something physically impossible.

In [ ]:
### START CODE HERE ### (~ 1 line of code)
df["implied_speed_kmh"] = None
### END CODE HERE ###

df.sort_values("implied_speed_kmh", ascending=False)[
    ["order_id", "zone", "distance_km", "duration_min", "implied_speed_kmh"]
].head(5)

**Expected output (top row):** an implied speed around **140 km/h** — not plausible for a scooter or bike in city traffic. Keep `implied_speed_kmh` — you'll use it again in Section 2.

---
# Section 2 — Logical / Business-Rule Checks

Statistics catch what's *unusual*. Logical checks catch what's *impossible* or *inconsistent* — using rules you know about the business, not the numbers alone. These checks are mostly provided; focus on reading and interpreting them.

### Exercise 12 — Timestamp Logic

A delivery can never be completed before the order was placed. Filter for any row where that's violated.

In [ ]:
### START CODE HERE ### (~ 1 line of code)
bad_timing = None
### END CODE HERE ###

print("Rows where actual_delivery_time is before order_time:", len(bad_timing))

**Expected output:** `0` — good, no timestamp is literally impossible. But that doesn't mean every timestamp is *plausible* (see Exercise 6 and 11).

### Exercise 13 — Speed Plausibility

This is provided in full — it applies a **business rule** (max realistic delivery speed in city traffic) on top of the `implied_speed_kmh` column you built in Exercise 11.

In [ ]:
MAX_PLAUSIBLE_KMH = 60  # generous upper bound for a scooter/bike/car in Cairo traffic
impossible_speed = df[df["implied_speed_kmh"] > MAX_PLAUSIBLE_KMH]
print("Rows implying an impossible delivery speed:", len(impossible_speed))
impossible_speed[["order_id", "distance_km", "duration_min", "implied_speed_kmh"]]

**Expected output:** `2` rows. Notice this catches a *subset* of what the IQR rule (Exercise 6) flagged — same underlying problem, found two different ways.

### Exercise 14 — Status Consistency Across Two Columns

`order_status` should never contradict `actual_delivery_time`: a `cancelled` order should have **no** delivery time, and a `delivered` order should **always** have one. Write both filters.

In [ ]:
### START CODE HERE ### (~ 2 lines of code)
cancelled_but_delivered = None
delivered_but_missing = None
### END CODE HERE ###

print("Cancelled orders that still show a delivery time:", len(cancelled_but_delivered))
print("Delivered orders missing a delivery time:", len(delivered_but_missing))

**Expected output:** `10` and `8`.

**Question:** what might cause each of these two inconsistencies operationally — not as a data bug, but as something that actually happened in the business? _(your answer here)_

### Exercise 15 — Egyptian Mobile Phone Format

A valid Egyptian mobile number starts with `010`, `011`, `012`, or `015`, followed by 8 digits (11 digits total). The regex pattern is provided — your part is applying it to every row.

In [ ]:
phone_pattern = re.compile(r"^01[0125]\d{8}$")

### START CODE HERE ### (~ 1 line of code)
df["valid_phone"] = None
### END CODE HERE ###

print("Invalid phone numbers:", (~df["valid_phone"]).sum(), f"({(~df['valid_phone']).mean():.1%})")
df.loc[~df["valid_phone"], "customer_phone"].head(10)

**Expected output:** **436 invalid** (14.4%).

**Question:** looking at the sample of invalid numbers above, can you spot more than one distinct *type* of formatting problem? _(your answer here)_

---
# Section 3 — Discussion: Is This Data Actually Useful for This Problem?

You've now run both statistical and logical checks. Step back and answer as a team — no code needed, just write your answers below.

**1.** Given everything you found, would you trust this dataset to train an ETA model **today**, or does it need cleaning first? Which issues are must-fix vs. nice-to-fix?

_(your answer here)_

**2.** The `Sheikh Zayed Extension` zone barely appears in the data (Exercise 4), has by far the highest missing-courier rate (Exercise 8), and its deliveries take significantly longer (Exercise 9). If you shipped a model trained on this data, what would it likely get wrong there — and why would collecting more of the *same kind* of data not fix that?

_(your answer here)_

**3.** Name one piece of data that **isn't in this dataset at all** that would meaningfully improve an ETA model (e.g. traffic conditions, restaurant prep time, weather).

_(your answer here)_

---
## Wrap-up

Submit this notebook (with all your code filled in and written answers completed) on Canvas.